In [1]:
%pip install "cupy-cuda13x[ctk]"
%pip install "pandas"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cupy as np
import pandas as pd

mnist_data = pd.read_csv(r'C:\Users\User\Desktop\mnist-quantization\train.csv')


In [3]:
def initialize_model(hidden1_size, hidden2_size):
    return {
        'W1': np.array([[(np.random.rand()*2-1)*0.01 for i in range(hidden1_size)] for i in range(784)]),
        'W2': np.array([[(np.random.rand()*2-1)*0.01 for i in range(hidden2_size)] for i in range(hidden1_size)]),
        'W3': np.array([[(np.random.rand()*2-1)*0.01 for i in range(10)] for i in range(hidden2_size)]),
        'b1': np.array([(np.random.rand()*2-1)*0.01 for i in range(hidden1_size)]),
        'b2': np.array([(np.random.rand()*2-1)*0.01 for i in range(hidden2_size)]),
        'b3': np.array([(np.random.rand()*2-1)*0.01 for i in range(10)]),
    }

def relu(values):
    return np.maximum(0, values)

def softmax(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    return np.exp(shifted_logits) / np.sum(np.exp(shifted_logits), axis=1, keepdims=True)

def forward(features, model):
    activations = {}
    layer_output = features.copy()
    for layer_index, (weights, biases) in enumerate([
        (model['W1'], model['b1']),
        (model['W2'], model['b2']),
    ], start=1):
        layer_output = np.dot(layer_output, weights) + biases
        activations[f'Z{layer_index}'] = layer_output
        layer_output = relu(layer_output)
        activations[f'A{layer_index}'] = layer_output
    layer_output = np.dot(layer_output, model['W3']) + model['b3']
    activations['Z3'] = layer_output
    activations['Yhat'] = softmax(layer_output)
    return activations

def loss(predictions, targets):
    return -np.sum(targets * np.log(predictions)) / predictions.shape[0]

def backward(features, targets, model):
    forward_values = forward(features, model)

    batch_size = features.shape[0]
    Z1 = forward_values['Z1']
    Z2 = forward_values['Z2']
    A1 = forward_values['A1']
    A2 = forward_values['A2']
    predictions = forward_values['Yhat']

    dZ3 = predictions - targets
    dW3 = np.dot(A2.T, dZ3) / batch_size
    db3 = np.mean(dZ3, axis=0)

    dA2 = np.dot(dZ3, model['W3'].T)
    dZ2 = dA2 * (Z2 > 0)
    dW2 = np.dot(A1.T, dZ2) / batch_size
    db2 = np.mean(dZ2, axis=0)

    dA1 = np.dot(dZ2, model['W2'].T)
    dZ1 = dA1 * (Z1 > 0)
    dW1 = np.dot(features.T, dZ1) / batch_size
    db1 = np.mean(dZ1, axis=0)

    weights = [model['W1'], model['W2'], model['W3']]
    biases = [model['b1'], model['b2'], model['b3']]
    weight_gradients = [dW1, dW2, dW3]
    bias_gradients = [db1, db2, db3]
    for layer_index in range(3):
        weights[layer_index] -= 0.01 * weight_gradients[layer_index]
        biases[layer_index] -= 0.01 * bias_gradients[layer_index]
    return loss(predictions, targets)


## 모델 A — 784 → 128 → 64 → 10


In [4]:
model_a = initialize_model(128, 64)


In [5]:
model_a_shuffled_df = mnist_data.sample(frac=1).reset_index(drop=True)
model_a_split_index = model_a_shuffled_df.shape[0] * 7 // 10
model_a_train_df = model_a_shuffled_df[:model_a_split_index].reset_index(drop=True)
model_a_test_df = model_a_shuffled_df[model_a_split_index:].reset_index(drop=True)
model_a_train_images = model_a_train_df[[f'pixel{i}' for i in range(mnist_data.shape[1]-1)]]
model_a_train_labels = model_a_train_df['label']
model_a_train_images = model_a_train_images / 255
model_a_train_images = np.asarray(model_a_train_images.to_numpy(), dtype=np.float64)
model_a_train_one_hot_labels = [[0]*10 for i in range(model_a_train_images.shape[0])]


In [6]:
for sample_index in range(model_a_train_images.shape[0]):
    model_a_train_one_hot_labels[sample_index][model_a_train_labels[sample_index]] = 1
model_a_train_one_hot_labels = np.array(model_a_train_one_hot_labels)

for epoch in range(1000):
    for batch_index in range(model_a_train_images.shape[0] // 128):
        batch_start = 128 * batch_index
        batch_end = 128 * (batch_index + 1)
        model_a_loss = backward(model_a_train_images[batch_start:batch_end],
                                model_a_train_one_hot_labels[batch_start:batch_end], model_a)
    if epoch % 10 == 0:
        print(f'{epoch}: {model_a_loss}')
print(f'loss: {model_a_loss}')


0: 2.301889902764667
10: 2.2962195830486114
20: 2.294400637349563
30: 1.530070330289727
40: 0.8422940937176687
50: 0.6999929712509684
60: 0.6063258296432477
70: 0.48802318326430794
80: 0.40053527386049737
90: 0.3402611835345897
100: 0.3038657143548405
110: 0.2715039763428434
120: 0.23849329608098038
130: 0.20989028720645397
140: 0.1856606076245554
150: 0.16573041100085817
160: 0.15048363359372602
170: 0.13766200336485213
180: 0.12671934086338055
190: 0.11694482539746238
200: 0.10833916232019315
210: 0.10034356070176732
220: 0.09293236242040309
230: 0.08546140820015696
240: 0.07773853939759058
250: 0.0702599653005217
260: 0.06306609774672157
270: 0.055814802133131745
280: 0.0488170866174559
290: 0.04236333513486536
300: 0.036054304535628016
310: 0.03086956330803114
320: 0.026590561896978587
330: 0.022709878020777823
340: 0.019942263420662833
350: 0.017845212164048566
360: 0.016266536083033196
370: 0.014961955053127347
380: 0.01374628363642929
390: 0.012781824456594204
400: 0.01192590837

In [7]:
model_a_test_images = model_a_test_df[[f'pixel{i}' for i in range(mnist_data.shape[1]-1)]]
model_a_test_labels = model_a_test_df['label']
model_a_test_images = model_a_test_images / 255
model_a_test_images = np.asarray(model_a_test_images.to_numpy(), dtype=np.float64)

model_a_misclassified_indices = []
model_a_test_predictions = forward(model_a_test_images, model_a)['Yhat']
for sample_index in range(model_a_test_images.shape[0]):
    if np.argmax(model_a_test_predictions[sample_index]) != model_a_test_labels[sample_index]:
        model_a_misclassified_indices.append(sample_index)

model_a_accuracy = 1 - len(model_a_misclassified_indices) / model_a_test_images.shape[0]
print('===== test ====')
print(f'classified: {model_a_test_images.shape[0]-len(model_a_misclassified_indices)}')
print(f'misclassified: {len(model_a_misclassified_indices)}')
print(f'accuracy: {model_a_accuracy}')


===== test ====
classified: 12197
misclassified: 403
accuracy: 0.968015873015873


In [13]:
print(f'W1: {model_a["W1"].nbytes}bytes, {model_a["W1"].nbytes / 1024}KB, {model_a["W1"].nbytes / 1024 / 1024}MB')
print(f'W2: {model_a["W2"].nbytes}bytes, {model_a["W2"].nbytes / 1024}KB, {model_a["W2"].nbytes / 1024 / 1024}MB')
print(f'W3: {model_a["W3"].nbytes}bytes, {model_a["W3"].nbytes / 1024}KB, {model_a["W3"].nbytes / 1024 / 1024}MB')
print(f'b1: {model_a["b1"].nbytes}bytes, {model_a["b1"].nbytes / 1024}KB, {model_a["b1"].nbytes / 1024 / 1024}MB')
print(f'b2: {model_a["b2"].nbytes}bytes, {model_a["b2"].nbytes / 1024}KB, {model_a["b2"].nbytes / 1024 / 1024}MB')
print(f'b3: {model_a["b3"].nbytes}bytes, {model_a["b3"].nbytes / 1024}KB, {model_a["b3"].nbytes / 1024 / 1024}MB')

W1: 802816bytes, 784.0KB, 0.765625MB
W2: 65536bytes, 64.0KB, 0.0625MB
W3: 5120bytes, 5.0KB, 0.0048828125MB
b1: 1024bytes, 1.0KB, 0.0009765625MB
b2: 512bytes, 0.5KB, 0.00048828125MB
b3: 80bytes, 0.078125KB, 7.62939453125e-05MB


In [18]:
import time

image = model_a_test_images[:1]

for _ in range(10):
    forward(image, model_a)
np.cuda.get_current_stream().synchronize()

times_ns = []
for _ in range(100):
    start_ns = time.perf_counter_ns()
    forward(image, model_a)
    np.cuda.get_current_stream().synchronize()
    times_ns.append(time.perf_counter_ns() - start_ns)

average_ms = sum(times_ns) / len(times_ns) / 1_000_000
print(f"이미지 1장 평균 추론 시간: {average_ms:.4f} ms")

이미지 1장 평균 추론 시간: 0.3037 ms


## 모델 B — 784 → 256 → 128 → 10

모델 A와 같은 계산 함수를 사용하며, 가중치와 데이터는 별도 변수에 보관합니다.


In [8]:
model_b = initialize_model(256, 128)


In [9]:
model_b_shuffled_df = mnist_data.sample(frac=1).reset_index(drop=True)
model_b_split_index = model_b_shuffled_df.shape[0] * 7 // 10
model_b_train_df = model_b_shuffled_df[:model_b_split_index].reset_index(drop=True)
model_b_test_df = model_b_shuffled_df[model_b_split_index:].reset_index(drop=True)
model_b_train_images = model_b_train_df[[f'pixel{i}' for i in range(mnist_data.shape[1]-1)]]
model_b_train_labels = model_b_train_df['label']
model_b_train_images = model_b_train_images / 255
model_b_train_images = np.asarray(model_b_train_images.to_numpy(), dtype=np.float64)
model_b_train_one_hot_labels = [[0]*10 for i in range(model_b_train_images.shape[0])]
for sample_index in range(model_b_train_images.shape[0]):
    model_b_train_one_hot_labels[sample_index][model_b_train_labels[sample_index]] = 1
model_b_train_one_hot_labels = np.array(model_b_train_one_hot_labels)


In [14]:
for epoch in range(1000):
    for batch_index in range(model_b_train_images.shape[0] // 128):
        batch_start = 128 * batch_index
        batch_end = 128 * (batch_index + 1)
        model_b_loss = backward(model_b_train_images[batch_start:batch_end],
                                model_b_train_one_hot_labels[batch_start:batch_end], model_b)
    if epoch % 5 == 0:
        print(f'{epoch}: {model_b_loss}')
print(f'loss: {model_b_loss}')


0: 0.8640090190956725
5: 0.697057052401081
10: 0.6005462114458476
15: 0.5260891724014477
20: 0.44488192336263116
25: 0.3744330537659121
30: 0.323326443832102
35: 0.28038131567306135
40: 0.24511031223466925
45: 0.21493984114718584
50: 0.18973895084166037
55: 0.16806431030244978
60: 0.14952550307247273
65: 0.13337913683409014
70: 0.11926377534942025
75: 0.10767489308715923
80: 0.09804924039156764
85: 0.08986333157172305
90: 0.08247349025026751
95: 0.07598258217452046
100: 0.07019575899879088
105: 0.06457412602132438
110: 0.05998745285608356
115: 0.055746832115055114
120: 0.05191211288806638
125: 0.048773861583395174
130: 0.04592895390477215
135: 0.04325668329286113
140: 0.04090543808961246
145: 0.03851648611521315
150: 0.03638815436154441
155: 0.03427455040890798
160: 0.03234637135521745
165: 0.030754285882469982
170: 0.029115866281760706
175: 0.02764813142041906
180: 0.02618768977224002
185: 0.024796575133365337
190: 0.023557843075795042
195: 0.022351892278314113
200: 0.0213021601781766

In [15]:
model_b_test_images = model_b_test_df[[f'pixel{i}' for i in range(mnist_data.shape[1]-1)]]
model_b_test_labels = model_b_test_df['label']
model_b_test_images = model_b_test_images / 255
model_b_test_images = np.asarray(model_b_test_images.to_numpy(), dtype=np.float64)

model_b_misclassified_indices = []
model_b_test_predictions = forward(model_b_test_images, model_b)['Yhat']
for sample_index in range(model_b_test_images.shape[0]):
    if np.argmax(model_b_test_predictions[sample_index]) != model_b_test_labels[sample_index]:
        model_b_misclassified_indices.append(sample_index)

model_b_accuracy = 1 - len(model_b_misclassified_indices) / model_b_test_images.shape[0]
print('===== test ====')
print(f'classified: {model_b_test_images.shape[0]-len(model_b_misclassified_indices)}')
print(f'misclassified: {len(model_b_misclassified_indices)}')
print(f'accuracy: {model_b_accuracy}')


===== test ====
classified: 12193
misclassified: 407
accuracy: 0.9676984126984127


In [16]:
print(f'W1: {model_b["W1"].nbytes}bytes, {model_b["W1"].nbytes / 1024}KB, {model_b["W1"].nbytes / 1024 / 1024}MB')
print(f'W2: {model_b["W2"].nbytes}bytes, {model_b["W2"].nbytes / 1024}KB, {model_b["W2"].nbytes / 1024 / 1024}MB')
print(f'W3: {model_b["W3"].nbytes}bytes, {model_b["W3"].nbytes / 1024}KB, {model_b["W3"].nbytes / 1024 / 1024}MB')
print(f'b1: {model_b["b1"].nbytes}bytes, {model_b["b1"].nbytes / 1024}KB, {model_b["b1"].nbytes / 1024 / 1024}MB')
print(f'b2: {model_b["b2"].nbytes}bytes, {model_b["b2"].nbytes / 1024}KB, {model_b["b2"].nbytes / 1024 / 1024}MB')
print(f'b3: {model_b["b3"].nbytes}bytes, {model_b["b3"].nbytes / 1024}KB, {model_b["b3"].nbytes / 1024 / 1024}MB')

W1: 1605632bytes, 1568.0KB, 1.53125MB
W2: 262144bytes, 256.0KB, 0.25MB
W3: 10240bytes, 10.0KB, 0.009765625MB
b1: 2048bytes, 2.0KB, 0.001953125MB
b2: 1024bytes, 1.0KB, 0.0009765625MB
b3: 80bytes, 0.078125KB, 7.62939453125e-05MB


In [17]:
import time

image = model_b_test_images[:1]

for _ in range(10):
    forward(image, model_b)
np.cuda.get_current_stream().synchronize()

times_ns = []
for _ in range(100):
    start_ns = time.perf_counter_ns()
    forward(image, model_b)
    np.cuda.get_current_stream().synchronize()
    times_ns.append(time.perf_counter_ns() - start_ns)

average_ms = sum(times_ns) / len(times_ns) / 1_000_000
print(f"이미지 1장 평균 추론 시간: {average_ms:.4f} ms")

이미지 1장 평균 추론 시간: 0.9295 ms


## INT8 정수 추론 공통 코드 및 모델 A

가중치와 활성화값은 INT8, 편향과 누산은 INT32입니다. 은닉층은 정수 고정소수점 재양자화를 거쳐 다음 층에 INT8을 전달합니다. 보정은 학습 데이터로만 수행합니다.


In [ ]:
import numpy as cpu_np


# np is the notebook's CuPy alias. These kernels never convert model values to float.
_INT8_HIDDEN = np.RawKernel(r'''
extern "C" __global__
void int8_hidden(const signed char* input, const signed char* weight,
                 const int* bias, signed char* output,
                 int batch, int input_width, int output_width,
                 long long multiplier, int shift) {
    int index = blockDim.x * blockIdx.x + threadIdx.x;
    int count = batch * output_width;
    if (index >= count) return;
    int row = index / output_width;
    int column = index % output_width;
    int accumulator = bias[column];
    for (int k = 0; k < input_width; ++k) {
        accumulator += (int)input[row * input_width + k] *
                       (int)weight[k * output_width + column];
    }
    // ReLU, then fixed-point requantization. The next layer receives INT8.
    if (accumulator < 0) accumulator = 0;
    long long rounded = (long long)accumulator * multiplier +
                        (1LL << (shift - 1));
    long long quantized = rounded >> shift;
    if (quantized > 127) quantized = 127;
    output[index] = (signed char)quantized;
}
''', 'int8_hidden')

_INT8_LOGITS = np.RawKernel(r'''
extern "C" __global__
void int8_logits(const signed char* input, const signed char* weight,
                 const int* bias, int* output,
                 int batch, int input_width, int output_width) {
    int index = blockDim.x * blockIdx.x + threadIdx.x;
    int count = batch * output_width;
    if (index >= count) return;
    int row = index / output_width;
    int column = index % output_width;
    int accumulator = bias[column];
    for (int k = 0; k < input_width; ++k) {
        accumulator += (int)input[row * input_width + k] *
                       (int)weight[k * output_width + column];
    }
    output[index] = accumulator;
}
''', 'int8_logits')


def _tensor_scale(values):
    maximum = float(np.max(np.abs(values)).get())
    if not cpu_np.isfinite(maximum):
        raise ValueError('Non-finite model parameter or activation')
    return maximum / 127.0 if maximum > 0 else 1.0


def quantize_input_images(normalized_images):
    """Convert already normalized [0, 1] images to signed INT8, once."""
    return np.clip(np.rint(normalized_images * 127.0), 0, 127).astype(np.int8)


def quantize_model_int8(model, calibration_images, batch_size=512):
    """Post-training calibration uses training images, never test images."""
    if not isinstance(calibration_images, np.ndarray):
        raise TypeError('Calibration images must be a CuPy array')
    if calibration_images.ndim != 2 or calibration_images.shape[1] != 784:
        raise ValueError('Expected normalized MNIST images with shape (N, 784)')
    if calibration_images.shape[0] == 0:
        raise ValueError('At least one calibration image is required')
    if batch_size <= 0:
        raise ValueError('batch_size must be positive')

    maxima = [0.0, 0.0]
    for start in range(0, calibration_images.shape[0], batch_size):
        values = forward(calibration_images[start:start + batch_size], model)
        maxima[0] = max(maxima[0], float(np.max(values['A1']).get()))
        maxima[1] = max(maxima[1], float(np.max(values['A2']).get()))

    input_scale = 1.0 / 127.0
    activation_scales = [input_scale] + [m / 127.0 if m > 0 else 1.0 for m in maxima]
    weight_scales = [_tensor_scale(model[f'W{layer}']) for layer in (1, 2, 3)]
    result = {}

    for index in range(3):
        layer = index + 1
        weight_scale = weight_scales[index]
        bias_scale = activation_scales[index] * weight_scale
        result[f'W{layer}'] = np.clip(
            np.rint(model[f'W{layer}'] / weight_scale), -127, 127
        ).astype(np.int8)
        rounded_bias = np.rint(model[f'b{layer}'] / bias_scale)
        if float(np.max(np.abs(rounded_bias)).get()) > cpu_np.iinfo(cpu_np.int32).max:
            raise OverflowError(f'INT32 bias overflow in layer {layer}')
        result[f'b{layer}'] = rounded_bias.astype(np.int32)
        maximum_bias = int(np.max(np.abs(result[f'b{layer}'].astype(np.int64))).get())
        maximum_accumulator = model[f'W{layer}'].shape[0] * 127 * 127 + maximum_bias
        if maximum_accumulator > cpu_np.iinfo(cpu_np.int32).max:
            raise OverflowError(f'INT32 accumulator may overflow in layer {layer}')

    fixed_shift = 30
    multipliers = []
    for index in (0, 1):
        ratio = activation_scales[index] * weight_scales[index] / activation_scales[index + 1]
        multiplier = round(ratio * (1 << fixed_shift))
        maximum_bias = int(np.max(np.abs(result[f'b{index + 1}'].astype(np.int64))).get())
        maximum_accumulator = model[f'W{index + 1}'].shape[0] * 127 * 127 + maximum_bias
        if not 0 < multiplier or maximum_accumulator * multiplier + (1 << (fixed_shift - 1)) > cpu_np.iinfo(cpu_np.int64).max:
            raise ValueError(f'Invalid fixed-point multiplier for layer {index + 1}')
        multipliers.append(multiplier)

    result['scales'] = cpu_np.asarray(activation_scales + weight_scales, dtype=cpu_np.float64)
    result['requant_multipliers'] = cpu_np.asarray(multipliers, dtype=cpu_np.int64)
    result['requant_shift'] = fixed_shift
    return result


def forward_int8(images_int8, quantized_model):
    """INT8 x INT8, INT32 accumulation/bias, integer-only hidden layers."""
    if not isinstance(images_int8, np.ndarray) or images_int8.dtype != np.int8:
        raise TypeError('Input must be an INT8 CuPy array')
    if images_int8.ndim == 1:
        images_int8 = images_int8[None, :]
    if images_int8.ndim != 2 or images_int8.shape[1] != quantized_model['W1'].shape[0]:
        raise ValueError('Input shape does not match the model')
    if images_int8.shape[0] == 0:
        raise ValueError('At least one image is required')

    current = images_int8
    threads = 256
    for index in (0, 1):
        layer = index + 1
        width = quantized_model[f'W{layer}'].shape[1]
        following = np.empty((current.shape[0], width), dtype=np.int8)
        element_count = current.shape[0] * width
        _INT8_HIDDEN(((element_count + threads - 1) // threads,), (threads,), (
            current, quantized_model[f'W{layer}'], quantized_model[f'b{layer}'], following,
            cpu_np.int32(current.shape[0]), cpu_np.int32(current.shape[1]), cpu_np.int32(width),
            quantized_model['requant_multipliers'][index],
            cpu_np.int32(quantized_model['requant_shift']),
        ))
        current = following

    width = quantized_model['W3'].shape[1]
    logits = np.empty((current.shape[0], width), dtype=np.int32)
    element_count = current.shape[0] * width
    _INT8_LOGITS(((element_count + threads - 1) // threads,), (threads,), (
        current, quantized_model['W3'], quantized_model['b3'], logits,
        cpu_np.int32(current.shape[0]), cpu_np.int32(current.shape[1]), cpu_np.int32(width),
    ))
    return logits


def int8_accuracy(images_int8, labels, quantized_model, batch_size=256):
    labels_gpu = np.asarray(labels.to_numpy(), dtype=np.int32)
    correct = 0
    for start in range(0, images_int8.shape[0], batch_size):
        end = start + batch_size
        logits = forward_int8(images_int8[start:end], quantized_model)
        correct += int(np.count_nonzero(np.argmax(logits, axis=1) == labels_gpu[start:end]).get())
    return correct / images_int8.shape[0]


def model_storage_bytes(model):
    return sum(model[name].nbytes for name in ('W1', 'W2', 'W3', 'b1', 'b2', 'b3'))


def quantized_storage_bytes(quantized_model):
    payload = model_storage_bytes(quantized_model)
    metadata = quantized_model['scales'].nbytes + quantized_model['requant_multipliers'].nbytes
    return payload + metadata

model_a_int8 = quantize_model_int8(model_a, model_a_train_images)
model_a_test_images_int8 = quantize_input_images(model_a_test_images)
model_a_int8_accuracy = int8_accuracy(
    model_a_test_images_int8, model_a_test_labels, model_a_int8
)
model_a_fp64_bytes = model_storage_bytes(model_a)
model_a_int8_bytes = quantized_storage_bytes(model_a_int8)
print(f'Model A FP64 accuracy: {model_a_accuracy:.4%}')
print(f'Model A INT8 accuracy: {model_a_int8_accuracy:.4%}')
print(f'Accuracy difference: {(model_a_int8_accuracy - model_a_accuracy) * 100:+.3f} percentage points')
print(f'FP64 parameters: {model_a_fp64_bytes:,} bytes')
print(f'INT8 parameters + scales: {model_a_int8_bytes:,} bytes')
print(f'Memory reduction: {(1 - model_a_int8_bytes / model_a_fp64_bytes) * 100:.2f}%')


## INT8 정수 추론 모델 B


In [ ]:
model_b_int8 = quantize_model_int8(model_b, model_b_train_images)
model_b_test_images_int8 = quantize_input_images(model_b_test_images)
model_b_int8_accuracy = int8_accuracy(
    model_b_test_images_int8, model_b_test_labels, model_b_int8
)
model_b_fp64_bytes = model_storage_bytes(model_b)
model_b_int8_bytes = quantized_storage_bytes(model_b_int8)
print(f'Model B FP64 accuracy: {model_b_accuracy:.4%}')
print(f'Model B INT8 accuracy: {model_b_int8_accuracy:.4%}')
print(f'Accuracy difference: {(model_b_int8_accuracy - model_b_accuracy) * 100:+.3f} percentage points')
print(f'FP64 parameters: {model_b_fp64_bytes:,} bytes')
print(f'INT8 parameters + scales: {model_b_int8_bytes:,} bytes')
print(f'Memory reduction: {(1 - model_b_int8_bytes / model_b_fp64_bytes) * 100:.2f}%')
